In [46]:
import numpy as np
import pandas as pd

In [47]:
df = pd.read_csv('diabetes.csv')

In [48]:
df.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [49]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [50]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [51]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [52]:
X = scaler.fit_transform(X)

In [53]:
X.shape

(768, 8)

In [54]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [55]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [56]:
model = Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [57]:
model.fit(X_train,y_train,batch_size=32,epochs=100,validation_data=(X_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3909 - loss: 0.7910 - val_accuracy: 0.4870 - val_loss: 0.7273
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5000 - loss: 0.7163 - val_accuracy: 0.6104 - val_loss: 0.6668
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6466 - loss: 0.6630 - val_accuracy: 0.7013 - val_loss: 0.6216
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6889 - loss: 0.6245 - val_accuracy: 0.7468 - val_loss: 0.5886
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7182 - loss: 0.5942 - val_accuracy: 0.7532 - val_loss: 0.5585
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7362 - loss: 0.5682 - val_accuracy: 0.7857 - val_loss: 0.5378
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7492 - loss: 0.5472 - val_accuracy: 0.7922 - val_loss: 0.5202
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7557 - loss: 0.5305 - val_accuracy: 0.8117

In [58]:
# 1. Selection of appropriate optimizer
# 2. No of nodes in a layer
# 3. how to select no. of layer
# 4. All in all one model

In [59]:
# pip install -U keras-tuner

**1. Selection of appropriate optimizer**

In [60]:
import kerastuner as kt

In [61]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(32,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  optimizer = hp.Choice('optimizer',values = ['adam','sgd','rmsprop','adadelta'])
  model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [62]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [63]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [64]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [65]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [66]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [67]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 92ms/step - accuracy: 0.7394 - loss: 0.5515 - val_accuracy: 0.7662 - val_loss: 0.5393
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.7541 - loss: 0.5299 - val_accuracy: 0.7727 - val_loss: 0.5228
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.7590 - loss: 0.5157 - val_accuracy: 0.7597 - val_loss: 0.5113
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.7590 - loss: 0.5047 - val_accuracy: 0.7597 - val_loss: 0.5043
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.7638 - loss: 0.4967 - val_accuracy: 0.7727 - val_loss: 0.4975
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7671 - loss: 0.4891 - val_accuracy: 0.7727 - val_loss: 0.4939
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7785 - loss: 0.4827 - val_accuracy: 0.7792 - val_loss: 0.4899
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7818 - loss: 0.4779 - val_accurac

**2. No of nodes in a layer**

In [70]:
def build_model(hp):
  model = Sequential()
  units = hp.Int('unitds',8,128,step=8)
  model.add(Dense(units=units,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [71]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials = 5,
                        directory = 'mydir',
                        project_name = 'Daibetes_Prediction')

In [72]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 14s


In [73]:
tuner.results_summary()

Results summary
Results in mydir/Daibetes_Prediction
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 4 summary
Hyperparameters:
unitds: 40
Score: 0.8051947951316833

Trial 2 summary
Hyperparameters:
unitds: 80
Score: 0.7857142686843872

Trial 3 summary
Hyperparameters:
unitds: 112
Score: 0.7857142686843872

Trial 0 summary
Hyperparameters:
unitds: 96
Score: 0.7727272510528564

Trial 1 summary
Hyperparameters:
unitds: 32
Score: 0.7662337422370911


In [74]:
tuner.get_best_hyperparameters()[0].values

{'unitds': 40}

In [75]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [78]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6)

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7736 - loss: 0.4991
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7769 - loss: 0.4857 
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7866 - loss: 0.4782 
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7801 - loss: 0.4722 
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7801 - loss: 0.4681 
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7785 - loss: 0.4642 
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7801 - loss: 0.4611 
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7801 - loss: 0.4583 
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7752 - loss: 0.4562 
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7818 - loss: 0.4544 
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7850 - loss: 0.4527 
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/

**Selection of no of layers**

In [95]:
def build_model(hp):
    model = Sequential()

    model.add(Dense(
        72,
        activation='relu',
        input_shape=(X_train.shape[1],)
    ))

    for i in range(hp.Int('num_layers', 1, 10)):
        model.add(Dense(72, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='rmsprop',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [100]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    directory='mydir',
    project_name='num_layers'
)

tuner.search(
    X_train,
    y_train,
    epochs=3,
    validation_data=(X_test, y_test)
)

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 10s


In [102]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [99]:
import shutil

shutil.rmtree('mydir', ignore_errors=True)

In [101]:
tuner.results_summary()

Results summary
Results in mydir/num_layers
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
num_layers: 3
Score: 0.8116883039474487

Trial 0 summary
Hyperparameters:
num_layers: 4
Score: 0.8051947951316833

Trial 2 summary
Hyperparameters:
num_layers: 2
Score: 0.8051947951316833


In [103]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3}

In [105]:
model  = tuner.get_best_models(num_models=1)[0]

In [106]:
model.fit(X_train,y_train,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.7736 - loss: 0.4657 - val_accuracy: 0.8182 - val_loss: 0.4626
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8013 - loss: 0.4465 - val_accuracy: 0.7857 - val_loss: 0.4776
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7932 - loss: 0.4327 - val_accuracy: 0.7922 - val_loss: 0.4996
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7948 - loss: 0.4336 - val_accuracy: 0.7662 - val_loss: 0.4657
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8046 - loss: 0.4229 - val_accuracy: 0.7727 - val_loss: 0.4769
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8046 - loss: 0.4183 - val_accuracy: 0.7922 - val_loss: 0.4740
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8029 - loss: 0.4083 - val_accuracy: 0.7727 - val_loss: 0.4752
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8062 - loss: 0.4016 - val_accuracy: 0.77

In [109]:
def build_model(hp):
  model = Sequential()
  counter = 0
  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
    if counter == 0:
      model.add(
          Dense(
              hp.Int('units'+str(i), min_value=8, max_value=128, step=8),
              activation=hp.Choice('activation'+ str(i),values=['relu','tanh','sigmoid']),
              input_dim=8
              )
          )
    else :
     model.add(
          Dense(
              hp.Int('units'+str(i), min_value=8, max_value=128, step=8),
              activation=hp.Choice('activation'+ str(i),values=['relu','tanh','sigmoid'])
              )
          )
     counter+=1

  model.add(Dense(1,activation='sigmoid'))
  model.compile(optimizer=hp.Choice('optimizer',values = ['rmsprop','adam','sgd','nadam','adadelta']),
                loss = 'binary_crossentropy',
                metrics=['accuracy'])
  return model

In [111]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials = 3,
                        directory = 'mydir',
                        project_name='final')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [112]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 07s]
val_accuracy: 0.8116883039474487

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 17s


In [113]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 10,
 'units0': 56,
 'activation0': 'relu',
 'optimizer': 'nadam',
 'units1': 24,
 'activation1': 'tanh',
 'units2': 120,
 'activation2': 'relu',
 'units3': 40,
 'activation3': 'sigmoid',
 'units4': 56,
 'activation4': 'relu',
 'units5': 72,
 'activation5': 'relu',
 'units6': 16,
 'activation6': 'relu',
 'units7': 112,
 'activation7': 'sigmoid',
 'units8': 96,
 'activation8': 'relu',
 'units9': 104,
 'activation9': 'relu'}

In [114]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 47 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [115]:
model.fit(X_train,y_train,epochs=200,initial_epoch=5,validation_data=(X_test,y_test))

Epoch 6/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.7785 - loss: 0.4718 - val_accuracy: 0.7792 - val_loss: 0.4720
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7752 - loss: 0.4649 - val_accuracy: 0.7727 - val_loss: 0.4893
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7736 - loss: 0.4679 - val_accuracy: 0.8052 - val_loss: 0.4654
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7964 - loss: 0.4537 - val_accuracy: 0.7922 - val_loss: 0.4594
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8078 - loss: 0.4558 - val_accuracy: 0.7662 - val_loss: 0.4805
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7899 - loss: 0.4572 - val_accuracy: 0.7922 - val_loss: 0.4598
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7948 - loss: 0.4448 - val_accuracy: 0.7987 - val_loss: 0.4571
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7932 - loss: 0.4331 - val_accuracy: 0.779